# Phase 1 Evidence-Pack Export (final deliverable)

Consolidates the whole Phase 1 evidence base into a **web-app-ready** export:

- `evidence_pack.json` - a single structured document (the data contract for the future
  visual web UI): project meta, the question, gate statuses, stage findings with evidence
  levels, the everything-in budget, the validation register, data sources, a layer manifest,
  and caveats.
- `layers/*.geojson` - map layers (boundary, routes, corridor, wayfinders, anchors).
- `executive_summary.md` - the human-readable pitch summary.

Assembled from the audited artifacts of notebooks 01-12. Descriptive; evidence levels and
caveats are carried through so the UI can show what is proven vs assumed. No funding claim.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd, geopandas as gpd, networkx as nx, osmnx as ox
from shapely.geometry import LineString, Point

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.spatial import audit
from spinelens.models import crossing as cr
from spinelens.models import pavilion as pv
from spinelens.models import budget as bg
from spinelens import validation as val
from spinelens.gate0b import utc_now_iso

DATA = PHASE1_ROOT / "data"; TABLES = PHASE1_ROOT / "outputs" / "tables"
EXPORTS = PHASE1_ROOT / "outputs" / "exports"; LAYERS = EXPORTS / "layers"
REPORTS = PHASE1_ROOT / "outputs" / "reports"; FIG_DIR = REPORTS / "evidence_pack_media"
for d in (EXPORTS, LAYERS, REPORTS, FIG_DIR): d.mkdir(parents=True, exist_ok=True)
ts = utc_now_iso()
ENVELOPE = 1_000_000
print("assembling evidence pack at", ts)

## Assemble map layers (GeoJSON)

In [ ]:
G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected(); largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
cl = {n: c for n, c in coords.items() if n in largest}
nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
fams = pd.read_csv(DATA / "route_families_phase1.csv")
def snap(nid): return audit.nearest_node(cl, (float(nodes.loc[nid,"latitude"]), float(nodes.loc[nid,"longitude"])))[0]

# routes (5 inbound) with RLI merged
rli = pd.read_csv(TABLES / "route_legibility_comparison.csv").set_index("route_family")
inbound = fams[fams["origin_node_id"] != fams["gateway_node_id"]]
route_rows = []
for _, f in inbound.iterrows():
    fid = f["route_family_id"]
    path = nx.shortest_path(G, snap(f["origin_node_id"]), snap(f["gateway_node_id"]), weight="length")
    geom = LineString([(coords[n][1], coords[n][0]) for n in path])
    route_rows.append({"route_family": fid, "origin": f["origin_node_id"], "priority": f["priority"],
                       "RLI": float(rli.loc[fid, "RLI"]) if fid in rli.index else None, "geometry": geom})
gpd.GeoDataFrame(route_rows, geometry="geometry", crs=4326).to_file(LAYERS / "routes.geojson", driver="GeoJSON")

# anchors (origins, gateway, crossing, onward)
anchors = pd.read_csv(DATA / "study_area_anchors_phase1.csv")
ga = gpd.GeoDataFrame(anchors, geometry=gpd.points_from_xy(anchors.longitude, anchors.latitude), crs=4326)
ga.to_file(LAYERS / "anchors.geojson", driver="GeoJSON")

# wayfinders (points with tier/role/content)
wc = pd.read_csv(TABLES / "wayfinder_content_pack.csv")
gw = gpd.GeoDataFrame(wc, geometry=gpd.points_from_xy(wc.lon, wc.lat), crs=4326)
gw.to_file(LAYERS / "wayfinders.geojson", driver="GeoJSON")

# corridor + boundary: copy/normalise from existing exports
gpd.read_file(EXPORTS / "tactical_corridor_segments.geojson").to_file(LAYERS / "corridor.geojson", driver="GeoJSON")
gpd.read_file(DATA / "interim" / "study_area_boundary_phase1.geojson").to_file(LAYERS / "study_boundary.geojson", driver="GeoJSON")
layer_files = sorted(p.name for p in LAYERS.glob("*.geojson"))
print("layers exported:", layer_files)

## Assemble findings, budget, validation and the master JSON

In [ ]:
# --- stage findings ---
routes_summary = rli[["RLI", "walk_time_min", "directness"]].reset_index().to_dict("records")
most, least = rli["RLI"].idxmax(), rli["RLI"].idxmin()

seg = pd.read_csv(TABLES / "tactical_corridor_segment_scores.csv")
corridor = {"trunk_km": round(seg.loc[seg.multiplicity >= 2, "length_m"].sum()/1000, 2),
            "spur_km": round(seg.loc[seg.multiplicity == 1, "length_m"].sum()/1000, 2),
            "severance_free": True, "nearest_major_road_m": 71.5,  # nb09 spatial check
            "note": "City-core spine avoids major roads; severance concentrated at the Nechells crossing."}

aadf = json.loads((DATA / "raw" / "dft_aadf" / "aadf_junction_points.json").read_text())
col = pd.read_csv(DATA / "raw" / "dft_stats19" / "collisions_near_junction_2020_2024.csv")
crossing = {
    "warrant": cr.crossing_warrant(30, 2, aadf["A4540 Middleway"]["aadf"]),
    "aadf_a4540": aadf["A4540 Middleway"]["aadf"], "cycles_a4540": aadf["A4540 Middleway"]["cycles"],
    "collisions_150m_2020_2024": int((col["dist_m"] <= 150).sum()),
    "serious_or_fatal_150m": int(col[(col["dist_m"] <= 150) & (col["collision_severity"].isin([1, 2]))].shape[0]),
    "phase1": "tactical enhancement now; full single-stage/Toucan upgrade is later-phase capital",
}

wc_tiers = wc["tier"].value_counts().to_dict()
# pavilion verdict recomputed with reduced (reversible) risks
pav_criteria = {"route_convergence": 0.90, "onward_access": 0.90, "demand": 0.95,
                "open_space": 0.50, "movement": 0.50, "deliverability": 0.55}  # deliverability up (reversible)
pav_weights = {"demand": 0.20, "route_convergence": 0.20, "onward_access": 0.20,
               "open_space": 0.15, "movement": 0.10, "deliverability": 0.15}
pav_score = cr.multi_criteria_rank({"pav": pav_criteria}, pav_weights)[0]["score"]
pav_risks = {"helipad": "medium", "licence": "medium", "footprint": "medium", "movement": "medium"}
pav_gate = pv.risk_gate(pav_risks)

# --- budget (everything in) recomputed from lines ---
bl = pd.read_csv(TABLES / "phase1_budget_lines.csv")
construction = {b: int(bl[b].sum()) for b in ("low", "central", "high")}
soft = bg.add_percentage(bg.add_percentage(construction, 13), 17)
total = bg.sum_costs([soft, bg.line_total(1, 20000, 45000, 80000)])
net = bg.apply_offset(total, bg.line_total(1, 10000, 25000, 40000))
budget = {"construction": construction, "total": total, "net_of_sponsorship": net,
          "envelope_gbp": ENVELOPE, "envelope_check": bg.envelope_check(net, ENVELOPE),
          "decisions": ["wooden reversible pavilion", "phased crossing (tactical now, full upgrade later)"]}

# --- validation + sources + gates ---
reg = pd.read_csv(TABLES / "phase1_validation_register.csv")
reg["gap"] = reg["required_level"] - reg["current_level"]
validation = {**val.register_summary(reg.to_dict("records")),
              "high_priority": reg[reg.priority == "high"]["item"].tolist()}
gates = pd.read_csv(DATA / "evidence_gate_status_phase1.csv")[["gate_id", "gate_name", "status"]].to_dict("records")
src = pd.read_csv(DATA / "source_acquisition_status_phase1.csv")
sources = src[src["evidence_level"].astype(str) != "0"][["source_id", "evidence_level", "can_support_funding_claim"]].to_dict("records")

pack = {
    "project": {"name": "SpineLens AI - Innovation Spine, Phase 1", "phase": "Phase 1 - Make It Visible",
                "generated_utc": ts, "envelope_gbp": ENVELOPE},
    "question": "Where does the city-to-B-KQ journey become illegible, and what low-cost interventions create the highest route clarity per pound?",
    "gates": gates,
    "stages": {
        "route_legibility": {"routes": routes_summary, "most_legible": most, "least_legible": least},
        "corridor": corridor,
        "wayfinders": {"count": int(len(wc)), "tiers": wc_tiers,
                       "model": "tiered; city-core light + pavilion hub; Nechells wayfinders carry hub-lite; civic sponsorship"},
        "crossing": crossing,
        "pavilion": {"score": pav_score, "verdict": pav_gate["verdict"], "risks": pav_risks,
                     "form": "wooden, reversible (demountable / meanwhile-use)"},
    },
    "budget": budget,
    "validation": validation,
    "data_sources": sources,
    "layers": [{"name": p.replace(".geojson", ""), "file": f"layers/{p}", "type": "geojson"} for p in layer_files],
    "caveats": [
        "Evidence is Level 3 (audited) at best; nothing is funding-grade until cross-check (L4) and field validation (L5).",
        "Unit costs are Level 1 indicative pending a quantity-surveyor costing.",
        "Anchors are provisional; OSM is volunteered data; the crossing/pavilion need consents.",
    ],
}
(EXPORTS / "evidence_pack.json").write_text(json.dumps(pack, indent=2), encoding="utf-8")
print("pavilion verdict (reduced risks):", pav_gate["verdict"], "| score", pav_score)
print("budget net central GBP{:,} | fits central={}".format(net["central"], budget["envelope_check"]["fits_central"]))
print("validation:", validation["open"], "open;", validation["field_validation_items"], "need field validation")
print("evidence_pack.json written with", len(pack["layers"]), "layers")

## Executive summary + 'evidence at a glance' visual

In [ ]:
def g(d): return f"GBP {d['low']:,} / {d['central']:,} / {d['high']:,}"
summary = [
    "# Innovation Spine - Phase 1 (SpineLens AI): Evidence Summary",
    "", f"Generated {ts}. Phase 1 'Make It Visible'. Descriptive evidence; no funding-facing claim.",
    "", "## The question", "", pack["question"],
    "", "## What the evidence shows", "",
    f"- **Legibility:** most legible approach {most}; least {least}. The city-core spine is severance-free "
    f"(nearest major road {corridor['nearest_major_road_m']} m), so the amber corridor can be near-continuous.",
    f"- **Crossing:** the Dartmouth barrier carries {crossing['aadf_a4540']:,} vehicles/day vs {crossing['cycles_a4540']} cycles, "
    f"with {crossing['collisions_150m_2020_2024']} injury collisions ({crossing['serious_or_fatal_150m']} serious/fatal) within 150 m (2020-2024). "
    "A zebra is not appropriate; a signalised crossing is warranted.",
    f"- **Wayfinders:** {len(wc)} tiered panels + the pavilion hub; Nechells wayfinders carry hub-lite content (no pavilion); civic-partner sponsorship part-funds it.",
    f"- **Pavilion:** wooden/reversible gateway hub; suitability {pav_score}, verdict '{pav_gate['verdict']}'.",
    "", "## Budget (everything in, vs GBP 1,000,000)", "",
    f"- Net of sponsorship: {g(net)}.",
    f"- Central {'fits' if budget['envelope_check']['fits_central'] else 'exceeds'} with headroom GBP {budget['envelope_check']['headroom_central']:,}.",
    f"- Decisions: {', '.join(budget['decisions'])}.",
    "", "## Validation frontier", "",
    f"- {validation['open']} open items; {validation['field_validation_items']} need field validation.",
    f"- High priority: {', '.join(validation['high_priority'])}.",
    "", "## Caveats", "",
] + [f"- {c}" for c in pack["caveats"]]
(EXPORTS / "executive_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\n".join(summary[:16]))

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
# routes RLI
r = rli.sort_values("RLI")
axes[0, 0].barh([i.replace("_to_ryder_gateway", "") for i in r.index], r["RLI"], color="#1a73e8")
axes[0, 0].set_xlim(0, 1); axes[0, 0].set_title("Route Legibility Index")
# budget vs envelope
bands = ["low", "central", "high"]
axes[0, 1].bar(bands, [net[b]/1000 for b in bands], color=["#34a853", "#1a73e8", "#fbbc04"])
axes[0, 1].axhline(ENVELOPE/1000, color="#d93025", ls="--", label="GBP 1,000,000")
axes[0, 1].set_ylabel("GBP thousands"); axes[0, 1].set_title("Budget (everything in, net)"); axes[0, 1].legend(fontsize=8)
# validation by priority
vp = reg["priority"].value_counts()
axes[1, 0].bar(vp.index, vp.values, color=[{"high": "#d93025", "medium": "#fbbc04", "low": "#34a853"}.get(p, "#9aa0a6") for p in vp.index])
axes[1, 0].set_title("Validation register by priority")
# data source evidence levels
sl = src[src["evidence_level"].astype(str) != "0"]["evidence_level"].value_counts().sort_index()
axes[1, 1].bar(sl.index.astype(str), sl.values, color="#5f6368")
axes[1, 1].set_xlabel("evidence level"); axes[1, 1].set_title("Acquired data sources by evidence level")
fig.suptitle("Phase 1 evidence at a glance", fontsize=15, weight="bold")
fig.tight_layout(); fig.savefig(FIG_DIR / "figV_evidence_at_a_glance.png", dpi=130, bbox_inches="tight"); plt.show()

## What this unlocks

The single, structured **evidence pack** the visual web app will consume: every map layer,
finding, cost and validation item in one JSON, with evidence levels and caveats carried
through. It is the Phase 1 deliverable - and the data contract for the UI to render an
amazing, easy-to-follow story from the city core to B-KQ.